In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/amarnathdj/audit-report-v1/audit_report_v1
/kaggle/input/datasets/amarnathdj/audit-report-v1/master_df_clean.parquet
/kaggle/input/datasets/amarnathdj/audit-report-v1/master_df_clean.pkl
/kaggle/input/datasets/amarnathdj/aligned-data/pickle files/june_2026_aligned_dataset.pkl
/kaggle/input/datasets/amarnathdj/aligned-data/pickle files/may_2024_aligned_dataset.pkl
/kaggle/input/datasets/amarnathdj/aligned-data/pickle files/june_2025_aligned_dataset.pkl
/kaggle/input/datasets/amarnathdj/aligned-data/pickle files/december_2023_aligned_dataset.pkl
/kaggle/input/datasets/amarnathdj/aligned-data/pickle files/november_2023_aligned_dataset.pkl
/kaggle/input/datasets/amarnathdj/aligned-data/pickle files/july_2024_aligned_dataset.pkl
/kaggle/input/datasets/amarnathdj/aligned-data/pickle files/april_2024_aligned_dataset.pkl
/kaggle/input/datasets/amarnathdj/aligned-data/pickle files/may_2025_aligned_dataset.pkl
/kaggle/input/datasets/amarnathdj/aligned-data/pickle files/may

# Batch Acoustic Feature Extraction
Loops over **every** aligned month in `master_df_clean.parquet`, extracts an expanded acoustic feature set for each `.wav` clip, aggregates clips up to one row per 3-minute rainfall window, and writes one `features_<month_name>.parquet` per dataset to `/kaggle/working/`.

Handles both audio schemes noted in the audit report (17-18 clips of 10s, or 57-60 clips of 3s) transparently, since aggregation just runs over however many clips are present in a window.

In [2]:
import os
import re
import gc
import glob
import warnings

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm

warnings.filterwarnings("ignore")

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
# Load the cleaned, consolidated label dataframe (already deduplicated and
# stripped of the invalid >100mm sensor-overflow rows, per audit_report_v1).
master_df = pd.read_parquet(
    "/kaggle/input/datasets/amarnathdj/audit-report-v1/master_df_clean.parquet"
)

print(master_df.shape)
master_df.head()


(30477, 5)


,timestamp,rainfall_mm,wav_count,wav_files,source_pickle
0,2026-06-09 17:38:43,0.00,17,[/kaggle/input/datasets/amarnathdj/june-2026-r...,june_2026_aligned_dataset.pkl
1,2026-06-09 17:41:49,0.00,17,[/kaggle/input/datasets/amarnathdj/june-2026-r...,june_2026_aligned_dataset.pkl
2,2026-06-09 17:41:50,0.00,17,[/kaggle/input/datasets/amarnathdj/june-2026-r...,june_2026_aligned_dataset.pkl
3,2026-06-09 17:44:57,0.00,17,[/kaggle/input/datasets/amarnathdj/june-2026-r...,june_2026_aligned_dataset.pkl
4,2026-06-09 17:48:05,0.71,17,[/kaggle/input/datasets/amarnathdj/june-2026-r...,june_2026_aligned_dataset.pkl


In [4]:
all_source_pickles = master_df["source_pickle"].unique()

print(f"Found {len(all_source_pickles)} datasets to process:")
for s in all_source_pickles:
    n = (master_df["source_pickle"] == s).sum()
    print(f"  - {s:45s} ({n} windows)")

Found 17 datasets to process:
  - june_2026_aligned_dataset.pkl                 (4959 windows)
  - may_2024_aligned_dataset.pkl                  (410 windows)
  - june_2025_aligned_dataset.pkl                 (3230 windows)
  - december_2023_aligned_dataset.pkl             (269 windows)
  - november_2023_aligned_dataset.pkl             (155 windows)
  - july_2024_aligned_dataset.pkl                 (295 windows)
  - april_2024_aligned_dataset.pkl                (326 windows)
  - may_2025_aligned_dataset.pkl                  (915 windows)
  - may_2026_aligned_dataset.pkl                  (1647 windows)
  - september_2024_aligned_dataset.pkl            (266 windows)
  - december_2024_aligned_dataset.pkl             (7946 windows)
  - feb_to_march_2026_aligned_dataset.pkl         (3068 windows)
  - october_2025_aligned_dataset.pkl              (684 windows)
  - november_2024_aligned_dataset.pkl             (1891 windows)
  - august_2025_aligned_dataset.pkl               (1344 windows)
  -

## Feature extraction (per audio clip)
Expanded relative to the original single-month version:
- Time-domain: zero-crossing rate, RMS (mean/std/max instead of just mean, since rain intensity shows up as energy *variability* within a clip, not just its average)
- Spectral shape: centroid, bandwidth, rolloff, **spectral flatness** (new — separates noise-like rain sound from tonal background noise), **spectral contrast** across 7 bands (new)
- **Chroma** (new — helps the model discount non-rain tonal/harmonic sounds, e.g. traffic, voices)
- MFCCs: still 13 coefficients (mean/std), plus **delta-MFCCs** (new — first-order time derivative, captures the *texture change* of the sound rather than a static snapshot)

In [5]:
N_MFCC = 13

def extract_clip_features(path):
    """Extract an acoustic feature dict from a single .wav clip.
    
    Handles various audio sample rates and corrupted/short clips gracefully.
    """
    try:
        y, sr = sf.read(path)
    except Exception:
        return None

    # Convert stereo to mono if necessary
    if y.ndim > 1:
        y = np.mean(y, axis=1)

    y = y.astype(np.float32)

    # Guard against very short/corrupt clips that break spectral feature windows
    min_len = 2048
    if len(y) < min_len:
        y = np.pad(y, (0, min_len - len(y)))

    features = {}

    # --- Time-domain ---
    features["zcr"] = librosa.feature.zero_crossing_rate(y).mean()

    rms = librosa.feature.rms(y=y)
    features["rms_mean"] = rms.mean()
    features["rms_std"] = rms.std()
    features["rms_max"] = rms.max()

    # --- Spectral shape ---
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    features["centroid_mean"] = centroid.mean()
    features["centroid_std"] = centroid.std()

    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features["bandwidth_mean"] = bandwidth.mean()

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features["rolloff_mean"] = rolloff.mean()

    flatness = librosa.feature.spectral_flatness(y=y)
    features["flatness_mean"] = flatness.mean()

    # Spectral contrast with error handling for low sample rates
    try:
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        for i in range(contrast.shape[0]):
            features[f"contrast_{i+1}_mean"] = contrast[i].mean()
    except librosa.util.exceptions.ParameterError:
        # For very low sample rates, use fewer bands or skip contrast
        nyquist = sr / 2
        if nyquist < 2000:  # Very low sample rate, skip contrast
            for i in range(1, 8):
                features[f"contrast_{i}_mean"] = 0.0
        else:
            # Try with reduced n_bands
            try:
                contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=3)
                for i in range(contrast.shape[0]):
                    features[f"contrast_{i+1}_mean"] = contrast[i].mean()
                # Pad to 7 bands
                for i in range(contrast.shape[0]+1, 8):
                    features[f"contrast_{i}_mean"] = 0.0
            except Exception:
                # Fall back to all zeros
                for i in range(1, 8):
                    features[f"contrast_{i}_mean"] = 0.0

    # --- Chroma ---
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features["chroma_mean"] = chroma.mean()

    # --- MFCCs + delta ---
    try:
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        mfcc_delta = librosa.feature.delta(mfcc)

        for i in range(N_MFCC):
            features[f"mfcc_{i+1}_mean"] = mfcc[i].mean()
            features[f"mfcc_{i+1}_std"] = mfcc[i].std()
            features[f"mfcc_delta_{i+1}_mean"] = mfcc_delta[i].mean()
    except Exception:
        # If MFCC fails, fill with zeros
        for i in range(N_MFCC):
            features[f"mfcc_{i+1}_mean"] = 0.0
            features[f"mfcc_{i+1}_std"] = 0.0
            features[f"mfcc_delta_{i+1}_mean"] = 0.0

    return features


In [6]:
# Quick sanity check — finds first row whose wav files actually exist on disk.
# Safe to run after removing previously-extracted datasets from Kaggle inputs.

sample_row = None
sample_path = None

for _, row in master_df.iterrows():
    candidate = row["wav_files"][0]
    if os.path.exists(candidate):
        sample_row = row
        sample_path = candidate
        break

if sample_path is None:
    print("WARNING: No attached dataset has audio files on disk.")
    print("Either attach a dataset or skip this sanity check.")
    print("Feature extraction will still run correctly for attached datasets.")
else:
    print(f"Using: {sample_path}")
    feat = extract_clip_features(sample_path)
    print(f"{len(feat)} features per clip")
    feat

Either attach a dataset or skip this sanity check.
Feature extraction will still run correctly for attached datasets.


## Per-window aggregation
Instead of only averaging clip-level features across a window (which is what the original notebook did), this aggregates with **both mean and std** across clips. The std across clips carries real signal here: a window with bursty/variable rain sound across its clips looks different from one with steady drizzle, even if the mean is similar.

In [7]:
def aggregate_window(clip_features_list):
    feat_df = pd.DataFrame(clip_features_list)
    agg = {}
    for col in feat_df.columns:
        agg[f"{col}_mean"] = feat_df[col].mean()
        agg[f"{col}_std"] = feat_df[col].std()
    agg["n_clips_used"] = len(feat_df)
    return agg


## Batch processing — one `features_<month_name>.parquet` per dataset
Iterates over every distinct `source_pickle` in `master_df`, extracts + aggregates features for that month only, and writes it out immediately (rather than holding everything in memory at once, which won't scale across ~2.5 years of data).

In [8]:
def month_name_from_pickle(source_pickle):
    return re.sub(r"_aligned_dataset\.pkl$", "", source_pickle)


def process_dataset(df_subset):
    features = []

    for _, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
        clip_features = []

        for path in row["wav_files"]:
            try:
                feat = extract_clip_features(path)
                clip_features.append(feat)
            except Exception:
                continue

        if len(clip_features) == 0:
            continue

        sample = aggregate_window(clip_features)
        sample["rainfall_mm"] = row["rainfall_mm"]
        sample["timestamp"] = row["timestamp"]
        sample["wav_count"] = row["wav_count"]

        features.append(sample)

    gc.collect()
    return pd.DataFrame(features)


In [9]:
summary = []

for source in all_source_pickles:
    month_name = month_name_from_pickle(source)
    print(f"\n=== Processing {month_name} ===")

    df_subset = master_df[master_df["source_pickle"] == source].copy()
    print(f"  windows to process: {len(df_subset)}")

    feature_df = process_dataset(df_subset)

    if feature_df.empty:
        print(f"  WARNING: no features extracted for {month_name}, skipping save.")
        continue

    # Classification target
    feature_df["rain"] = (feature_df["rainfall_mm"] > 0).astype(int)

    out_path = f"{OUTPUT_DIR}/features_{month_name}.parquet"
    feature_df.to_parquet(out_path, index=False)

    print(f"  saved -> {out_path}  shape={feature_df.shape}")

    summary.append({
        "month": month_name,
        "n_windows": len(feature_df),
        "n_features": feature_df.shape[1],
        "n_rain_windows": int(feature_df["rain"].sum()),
        "output_path": out_path,
    })

    del feature_df
    gc.collect()

summary_df = pd.DataFrame(summary)
summary_df



=== Processing june_2026 ===
  windows to process: 4959


100%|██████████| 4959/4959 [00:04<00:00, 1136.27it/s]


  saved -> /kaggle/working/features_june_2026.parquet  shape=(4959, 7)

=== Processing may_2024 ===
  windows to process: 410


100%|██████████| 410/410 [00:00<00:00, 1184.30it/s]


  saved -> /kaggle/working/features_may_2024.parquet  shape=(410, 7)

=== Processing june_2025 ===
  windows to process: 3230


...

100%|██████████| 3024/3024 [00:02<00:00, 1165.31it/s]


  saved -> /kaggle/working/features_jan_2025.parquet  shape=(3024, 7)

=== Processing january_2024 ===
  windows to process: 48


100%|██████████| 48/48 [00:00<00:00, 563.03it/s]


  saved -> /kaggle/working/features_january_2024.parquet  shape=(48, 7)


,month,n_windows,n_features,n_rain_windows,output_path
0,june_2026,4959,7,259,/kaggle/working/features_june_2026.parquet
1,may_2024,410,7,374,/kaggle/working/features_may_2024.parquet
2,june_2025,3230,7,348,/kaggle/working/features_june_2025.parquet
3,december_2023,269,7,202,/kaggle/working/features_december_2023.parquet
4,november_2023,155,7,95,/kaggle/working/features_november_2023.parquet
5,july_2024,295,7,76,/kaggle/working/features_july_2024.parquet
6,april_2024,326,7,110,/kaggle/working/features_april_2024.parquet
7,may_2025,915,7,232,/kaggle/working/features_may_2025.parquet
8,may_2026,1647,7,63,/kaggle/working/features_may_2026.parquet
9,september_2024,266,7,171,/kaggle/working/features_september_2024.parquet


## Processing Status
Check what's been extracted so far.

In [10]:
import glob
import os

feature_files = sorted(glob.glob(f"{OUTPUT_DIR}/features_*.parquet"))
feature_files = [f for f in feature_files if 'combined' not in f]

print(f"✓ {len(feature_files)} monthly feature files on disk:")
for f in feature_files:
    df = pd.read_parquet(f)
    month = os.path.basename(f).replace('features_', '').replace('.parquet', '')
    rain_pct = 100 * df['rain'].sum() / len(df)
    print(f"  {month:20s} {len(df):5d} windows ({rain_pct:5.1f}% rain)")

total_windows = sum([len(pd.read_parquet(f)) for f in feature_files])
print(f"\nTotal: {total_windows} windows across all months")

✓ 17 monthly feature files on disk:
  april_2024             326 windows ( 33.7% rain)
  august_2025           1344 windows ( 99.0% rain)
  december_2023          269 windows ( 75.1% rain)
  december_2024         7946 windows (  0.4% rain)
  feb_to_march_2026     3068 windows ( 99.9% rain)
  jan_2025              3024 windows (  6.3% rain)
  january_2024            48 windows ( 22.9% rain)
  july_2024              295 windows ( 25.8% rain)
  june_2025             3230 windows ( 10.8% rain)
  june_2026             4959 windows (  5.2% rain)
  may_2024               410 windows ( 91.2% rain)
  may_2025               915 windows ( 25.4% rain)
  may_2026              1647 windows (  3.8% rain)
  november_2023          155 windows ( 61.3% rain)
  november_2024         1891 windows (  6.0% rain)
  october_2025           684 windows ( 56.7% rain)
  september_2024         266 windows ( 64.3% rain)

Total: 30477 windows across all months


### Batch Processing Workflow

Your data processing is **incremental and safe**:

1. **Run 1**: Upload datasets A-E → run extraction → creates `features_A.parquet` ... `features_E.parquet`
2. **Run 2**: Delete A-E from Kaggle inputs, upload F-H → run extraction → creates `features_F.parquet` ... `features_H.parquet`
3. **Combine**: The cell below reads **all** `features_*.parquet` files currently in `/kaggle/working/`, so it automatically includes A-H (even though A-E are no longer in the input datasets).

**Each monthly `.parquet` is immutable once written**. The `features_all_combined.parquet` is rebuilt from scratch each time you run the combine cell, so it always reflects everything that's been extracted so far.

## Optional: combine all monthly parquet files into one
Handy for the actual model training step, while still keeping the per-month files for auditing or targeted re-processing of a single month.

In [11]:
all_feature_files = sorted(glob.glob(f"{OUTPUT_DIR}/features_*.parquet"))
print(f"Found {len(all_feature_files)} monthly feature files")

combined_df = pd.concat(
    [pd.read_parquet(f) for f in all_feature_files if "combined" not in f],
    ignore_index=True
)

combined_out_path = f"{OUTPUT_DIR}/features_all_combined.parquet"
combined_df.to_parquet(combined_out_path, index=False)

print(f"Combined shape: {combined_df.shape}")
print(f"Saved -> {combined_out_path}")


Found 17 monthly feature files
Combined shape: (30477, 7)
Saved -> /kaggle/working/features_all_combined.parquet
